# Prototyping: CIFAR-10 in plain PyTorch

## The task

Classify 32x32 colour images into 10 classes: plane, car, bird, cat, deer, dog, frog, horse,
ship and truck. CIFAR-10 has 50,000 training images and 10,000 test images. It is small enough to
train in minutes on a laptop GPU, and hard enough that design choices show in the results.

## The questions

Getting one model to train is only the start. The research is the questions that come after it:

- **Model configuration.** How wide should the network be? Does a deeper one help?
- **Optimisation.** Which learning rate, weight decay and batch size? How many epochs?
- **Data.** Does augmentation help, and by how much?
- **Is it real?** Is a 0.5% gain a better model, or a lucky seed?
- **Beyond accuracy.** Which classes fail? Can we trust the model's confidence? What did the run
  cost?

Each of these is answered by many runs, compared side by side. This notebook gets the first run
working.

## Why a notebook first

Start every project in a notebook. Everything is in front of you, any cell can be changed and
re-run, and you can look at the data and the model as you go. `with_tooling/` is what this
notebook grows into once the questions above need hundreds of runs, on a cluster, on several
GPUs.

**Reading the comments.** A line marked `# ->` is boilerplate. In `with_tooling/` it is either
done for you by a library, or it moves to a fixed place. The tag says which library takes it
over, and where to find it:

| Tag | Takes over | In `with_tooling/` |
|---|---|---|
| `[Lightning]` | the loop, devices, precision, checkpoints | `data.py`, `module.py`, `train.py` |
| `[wandb]` | logging, curves, comparing runs | `train.py`, `module.py` |
| `[torchmetrics]` | metrics that stay correct across batches and GPUs | `module.py`, `metrics.py` |
| `[OmegaConf]` | settings, per experiment and per machine | `configs/`, `config.py` |

The code without a tag is the research itself: the data, the model and the loss.

In [ ]:
import random
import time
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchvision.utils import make_grid

## Settings

Every hyperparameter of the run, in one place. Answering a question above means changing one of
these, re-running, and comparing.

In [ ]:
# -> [OmegaConf] configs/base.yaml, with per-machine values from configs/where/ (config.py).
# -> [wandb] the whole config is saved with every run, so runs can be compared by setting.
cfg = SimpleNamespace(
    data_dir="data/",
    out_dir="outputs/prototyping/",
    val_size=5000,
    batch_size=128,
    num_workers=4,
    augment=True,
    width=64,
    epochs=10,
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    # bf16 autocast needs an NVIDIA GPU from Ampere on (RTX 30xx and later): off on MPS and CPU.
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    seed=0,
)

In [ ]:
# -> [Lightning] L.seed_everything(cfg.seed) in train.py.
def seed_everything(seed: int) -> None:
    """Seed every random number generator the run touches."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything(cfg.seed)
# -> [Lightning] Trainer(accelerator="auto", devices="auto") in train.py picks the device(s).
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")

## Data

50,000 training images, of which 5,000 are held out for validation, and 10,000 test images.
Validation reads the same files as training, without the augmentation.

In [ ]:
# -> [Lightning] CIFAR10DataModule in data.py: this same code, split into hooks.
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
CLASSES = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")


def cifar_transforms(augment: bool) -> transforms.Compose:
    """Random crop and flip when augmenting, then normalise."""
    steps = []
    if augment:
        steps += [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]
    steps += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
    return transforms.Compose(steps)


# ##############################################################################################
# NOTE: evaluation never augments, unless it is deliberate.
#
# Validation and test must be deterministic: the same model has to get the same score every
# time, or comparing runs and picking the best checkpoint mean nothing. If you do want a
# stochastic evaluation (test-time augmentation, say), make it intentional and controlled: its
# own setting, a fixed seed, and reported as such.
# ##############################################################################################
train_tf, eval_tf = cifar_transforms(cfg.augment), cifar_transforms(augment=False)

# -> [Lightning] prepare_data() downloads once; setup() builds the datasets and the split.
train_full = CIFAR10(cfg.data_dir, train=True, download=True, transform=train_tf)
val_full = CIFAR10(cfg.data_dir, train=True, transform=eval_tf)
test_set = CIFAR10(cfg.data_dir, train=False, download=True, transform=eval_tf)

# A fixed split, so every run validates on the same images.
perm = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(cfg.seed))
train_set = Subset(train_full, perm[cfg.val_size :].tolist())
val_set = Subset(val_full, perm[: cfg.val_size].tolist())


# -> [Lightning] train_dataloader(), val_dataloader() and test_dataloader().
def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    """Batches of `cfg.batch_size`, loaded by `cfg.num_workers` worker processes."""
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=device.type == "cuda",
        persistent_workers=cfg.num_workers > 0,
    )


train_loader = make_loader(train_set, shuffle=True)
val_loader = make_loader(val_set, shuffle=False)
test_loader = make_loader(test_set, shuffle=False)
print(f"train {len(train_set)} | val {len(val_set)} | test {len(test_set)} images")

Always look at a batch before training on it: it catches broken transforms and shuffled
labels.

In [ ]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

grid = make_grid(images[:32], nrow=8)
grid = grid * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)  # un-normalise
plt.figure(figsize=(8, 4.5))
plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
plt.axis("off")
plt.show()
print(" ".join(CLASSES[i] for i in labels[:8]))

## Model

Three blocks of two 3x3 convolutions, each block followed by a max-pool (32 -> 16 -> 8 -> 4
pixels), then a global average pool and a linear classifier.

In [ ]:
# -> with_tooling/model.py, unchanged: the model is research, not boilerplate.
def conv_block(c_in: int, c_out: int) -> nn.Sequential:
    """Two 3x3 conv-BN-ReLU layers, then a 2x2 max-pool that halves height and width."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class SmallCNN(nn.Module):
    """(B, 3, 32, 32) images -> (B, num_classes) logits."""

    def __init__(self, num_classes: int = 10, width: int = 64) -> None:
        """Build the three blocks and the classifier head."""
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, width),
            conv_block(width, 2 * width),
            conv_block(2 * width, 4 * width),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(4 * width, num_classes),
        )

    def forward(self, x: Tensor) -> Tensor:
        """Features, global average pool, then logits."""
        return self.head(self.features(x))

Re-run from here to start a fresh run: the model and the optimiser carry their state from
one run of the training cell to the next.

In [ ]:
# -> [Lightning] Classifier.__init__() in module.py builds the model; Lightning moves it
#    to the device.
model = SmallCNN(width=cfg.width).to(device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
with torch.no_grad():
    print(model(images[:2].to(device)).shape)  # a quick shape check: (2, 10)

# -> [Lightning] configure_optimizers() in module.py. The number of steps for the schedule
#    comes from self.trainer.estimated_stepping_batches.
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=cfg.lr,
    momentum=cfg.momentum,
    weight_decay=cfg.weight_decay,
    nesterov=True,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=cfg.lr, total_steps=cfg.epochs * len(train_loader)
)

## One pass over the data

Loss and accuracy are kept as sums and counts, and divided once at the end. A mean of per-batch
means would weight a short last batch like a full one.

In [ ]:
# -> [Lightning] the loop is Trainer.fit() in train.py. What you write is training_step()
#    in module.py: the forward pass and the loss, nothing else.
def train_one_epoch(loader: DataLoader) -> dict[str, float]:
    """Train for one epoch, stepping the LR schedule every batch."""
    model.train()  # -> [Lightning] done for you
    # -> [torchmetrics] MeanMetric and MulticlassAccuracy in module.py keep these sums.
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    total = 0
    for x, y in loader:
        # -> [Lightning] batches arrive already on the device.
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        # -> [Lightning] Trainer(precision="bf16-mixed") in train.py.
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=cfg.bf16):
            logits = model(x)
            loss = F.cross_entropy(logits, y)

        # -> [Lightning] all four lines: training_step() returns the loss, Lightning does the
        #    rest (and gradient clipping or accumulation, with one Trainer argument each).
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # -> [torchmetrics] metric.update(logits, y); compute() divides once, across GPUs too.
        loss_sum += loss.detach() * len(y)
        correct += (logits.argmax(dim=-1) == y).sum()
        total += len(y)
    # -> [wandb] self.log("train/loss", ...) in module.py sends each value to the run.
    return {"loss": (loss_sum / total).item(), "acc": (correct / total).item()}


# -> [Lightning] validation_step() and test_step() in module.py. Lightning switches to
#    model.eval() and torch.no_grad() for you.
@torch.no_grad()
def evaluate(loader: DataLoader) -> dict[str, float]:
    """Mean loss and accuracy over every image the loader yields, without training."""
    model.eval()
    # -> [torchmetrics] one set of metrics per split, in module.py.
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)
    total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=cfg.bf16):
            logits = model(x)
        loss_sum += F.cross_entropy(logits.float(), y, reduction="sum")
        correct += (logits.argmax(dim=-1) == y).sum()
        total += len(y)
    return {"loss": (loss_sum / total).item(), "acc": (correct / total).item()}

## Train

Validate after every epoch, and keep the checkpoint with the best validation accuracy.

In [ ]:
# -> [wandb] every run gets its own directory and its own page, with nothing to name by hand.
out_dir = Path(cfg.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
# -> [wandb] every logged value is kept, per run, for good.
history = {f"{s}_{m}": [] for s in ("train", "val") for m in ("loss", "acc")}
best_acc, best_path = 0.0, out_dir / "best.pt"

# -> [Lightning] trainer.fit(model, datamodule=datamodule) in train.py runs this whole loop.
for epoch in range(1, cfg.epochs + 1):
    start = time.perf_counter()
    train = train_one_epoch(train_loader)
    val = evaluate(val_loader)
    for split, results in (("train", train), ("val", val)):
        for name, value in results.items():
            history[f"{split}_{name}"].append(value)
    # -> [Lightning] the progress bar; [wandb] live curves while the run goes.
    print(
        f"epoch {epoch:3d}/{cfg.epochs} | "
        f"train loss {train['loss']:.3f} acc {train['acc']:.3f} | "
        f"val loss {val['loss']:.3f} acc {val['acc']:.3f} | "
        f"{time.perf_counter() - start:.1f}s"
    )
    # -> [Lightning] ModelCheckpoint(monitor="val/acc", mode="max") in train.py.
    if val["acc"] > best_acc:
        best_acc = val["acc"]
        torch.save({"model": model.state_dict(), "epoch": epoch, "cfg": vars(cfg)}, best_path)

In [ ]:
# -> [wandb] the dashboard: every curve of every run, side by side, nothing to plot by hand.
epochs = range(1, len(history["train_loss"]) + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(10, 4))
for split in ("train", "val"):
    ax_loss.plot(epochs, history[f"{split}_loss"], label=split)
    ax_acc.plot(epochs, history[f"{split}_acc"], label=split)
ax_loss.set(xlabel="epoch", ylabel="loss", title="Loss")
ax_acc.set(xlabel="epoch", ylabel="accuracy", title="Accuracy")
ax_loss.legend()
ax_acc.legend()
fig.tight_layout()
plt.show()

## Test

Once, at the end, on the checkpoint that was best on validation.

In [ ]:
# -> [Lightning] trainer.test(model, datamodule=datamodule, ckpt_path="best") in train.py.
model.load_state_dict(torch.load(best_path, map_location=device)["model"])
test = evaluate(test_loader)
print(f"test (best val checkpoint) | loss {test['loss']:.3f} acc {test['acc']:.3f}")

## Where the notebook stops

One run works. Now go back to the questions at the top, and the notebook starts to get in the
way:

- **Comparing runs.** Re-run with another learning rate and the curves above are gone. Comparing
  ten settings over three seeds means 30 runs, each with its config and its curves.
  → **wandb**
- **Changing settings.** Each question means editing the settings cell, and remembering what
  you edited. A cluster job runs a script, not a notebook, with settings per experiment and per
  machine. → **OmegaConf**
- **More than accuracy.** Per-class accuracy and calibration are more sums to keep by hand, and
  on several GPUs each process only counts its own share of the data. → **torchmetrics**
- **The boilerplate.** Half of this notebook is device moves, `train()`/`eval()`, `zero_grad`,
  autocast and checkpoints, and it is the same in every project. → **Lightning**

`with_tooling/` is this notebook, with every `# ->` line handed over.